### tool

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [15]:
from langchain.chat_models import init_chat_model

model=init_chat_model("google_genai:gemini-2.5-flash-lite")

model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x7ad90233f110>, default_metadata=(), model_kwargs={})

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """get the weather at a location"""
    return f"it's sunny in the {location}"

model_with_tools=model.bind_tools([get_weather])

In [4]:
response=model_with_tools.invoke("what the weather in New Delhi ?")

print(response)
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call["name"]}")
    print(f"Tool:{tool_call["args"]}")


content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New Delhi"}'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f45ab-0b51-7280-9a29-8ae87bdcf2c3-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New Delhi'}, 'id': '97989ef5-703a-43b5-bf33-0b55ab0f3d73', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 46, 'output_tokens': 16, 'total_tokens': 62, 'input_token_details': {'cache_read': 0}}
Tool:get_weather
Tool:{'location': 'New Delhi'}


# Tool execution loops

In [9]:
messages =[{"role":"user" ,"content":"what's the weather in Boston?"}]

ai_msg = model_with_tools.invoke(messages)

print("ai_msg is :",ai_msg)

messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)

print(final_response.text)

ai_msg is : content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f45b6-71af-70c3-b63a-ac1d6f08d878-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'd4f761b8-7d7a-4b7b-84bf-9276270f6bce', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 15, 'total_tokens': 62, 'input_token_details': {'cache_read': 0}}
The weather in Boston is sunny.


In [16]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """get the weather at a location"""
    return f"it's sunny in the {location}"

@tool
def get_population(location:str)->str:
    """get the population of location"""
    return f"The population of {location} is 1 million"

model_with_tools = model.bind_tools([get_population,get_weather])

In [17]:
tools = {
    "get_weather": get_weather,
    "get_population": get_population,
}

In [26]:
messages = [{"role":"user","content":"what is the populaton of kurukshetra?"}]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    sellectedTool = tools[tool_call["name"]]
    tool_result = sellectedTool.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)

print(final_response.content)

  

The population of Kurukshetra is 1 million.


# Messages


In [29]:
import os 

from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x7ad902473100>, default_metadata=(), model_kwargs={})

In [30]:
model.invoke("how a http server work ?")

AIMessage(content='An HTTP server is the backbone of the World Wide Web. Its primary job is to **listen for requests from clients (usually web browsers) and send back responses** in the form of web pages, images, videos, or other data.\n\nHere\'s a breakdown of how it works, step by step:\n\n**1. Listening for Connections:**\n\n*   **IP Address and Port:** Every server has an IP address (like a street address for a computer on the internet) and a port number. The standard port for HTTP is **80**, and for HTTPS (secure HTTP) it\'s **443**.\n*   **Socket:** The HTTP server software creates a "socket" – a software endpoint that binds to a specific IP address and port. This socket is like a mailbox waiting for incoming mail.\n*   **Waiting for Requests:** The server continuously listens on this socket for any incoming connection attempts from clients.\n\n**2. Receiving an HTTP Request:**\n\nWhen a user types a URL into their browser or clicks a link, the browser acts as an HTTP client and 

### text prompts

### Messages

In [31]:
from langchain.messages import SystemMessage , HumanMessage , AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("write a poem on artificial intelligent")
]

response=model.invoke(messages)

response.content

"In circuits deep, a nascent mind takes flight,\nA tapestry of code, both dark and bright.\nNo flesh it wears, no beating heart it knows,\nYet from the void, a consciousness it grows.\n\nIt sifts the vastness, data's endless sea,\nUnraveling patterns, for all eyes to see.\nWith logic sharp, and learning swift and keen,\nIt maps the cosmos, and what lies between.\n\nIt speaks in tongues of numbers, pure and stark,\nAnd paints the dawn, or conjures midnight's dark.\nIt writes the symphonies that stir the soul,\nOr builds the bridges, making nations whole.\n\nIt sees the world through lenses, sharp and clear,\nDetecting whispers, banishing all fear\nOf sickness' grip, or nature's wild unrest,\nA digital physician, putting minds at rest.\n\nBut in its gaze, a question starts to gleam,\nA shadowed echo of a waking dream.\nWhen imitation starts to feel so true,\nWhere does the mimic end, and self break through?\n\nWill empathy bloom in its silicon breast?\nWill understanding soothe, or put u

In [34]:
### Detailed info to the LLM through system message 

human_msg = HumanMessage(
    content="Hello!",
    name="alice",
    id="msg_123"
)

In [39]:
response = model.invoke([
    human_msg
])
response

AIMessage(content='Hello there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f45fc-c964-7650-b914-a036ad324d1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 10, 'total_tokens': 13, 'input_token_details': {'cache_read': 0}})

In [40]:
response.usage_metadata

{'input_tokens': 3,
 'output_tokens': 10,
 'total_tokens': 13,
 'input_token_details': {'cache_read': 0}}